In [ ]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from src.pipeline import img_pipeline

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff', '.svg'}


def convert(source_folder: Path, output_folder: Path):
    """Convert every image of source_folder to RGB JPG, transparency flattened on white."""
    output_folder.mkdir(parents=True, exist_ok=True)
    converted_count = 0
    error_count = 0

    for file_path in source_folder.iterdir():
        # Windows downloads come with ':Zone.Identifier' side files
        if ':Zone.Identifier' in file_path.name or not file_path.is_file():
            continue
        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(file_path) as img:
                if img.mode in ('RGBA', 'LA', 'P'):
                    rgb_img = Image.new('RGB', img.size, (255, 255, 255))
                    if img.mode == 'P':
                        img = img.convert('RGBA')
                    rgb_img.paste(img, mask=img.split()[-1])
                    img = rgb_img
                elif img.mode != 'RGB':
                    img = img.convert('RGB')

                img.save(output_folder / f"{file_path.stem}.jpg", 'JPEG', quality=95)
                converted_count += 1
        except Exception as e:
            error_count += 1
            print(f"Failed on {file_path.name}: {e}")

    print(f"Converted {converted_count} images to {output_folder}, {error_count} errors")

In [ ]:
# Trashcans: convert(Path("google_lens_trashcan_dl"), Path("google_lens_trashcan_dl_jpg"))
convert(Path("datasets/raw/red balls human yahoo"), Path("datasets/cleaned/red_balls_human_yahoo_jpg"))

In [2]:
img_paths = [*Path("tom_trashcans").glob("*"), *Path("google_lens_trashcan_dl_jpg").glob("*")]

In [ ]:
from src.detection import YOLODetector
from src.segmentation import SAMSegmenter

yolo_detector = YOLODetector("yolo11n.pt", conf=0.25)
segmenter = SAMSegmenter()


def detect_by_label(img_path: Path) -> dict[str, list[dict]]:
    """Adapt YOLODetector.detect to the {label: [detections]} shape img_pipeline expects."""
    det = yolo_detector.detect(img_path)
    return {det['label']: [det]} if det else {}

In [ ]:
(DET_OUTPUT := Path("auto_trashcan_det")).mkdir(exist_ok=True)
(SEG_OUTPUT := Path("auto_trashcan_seg")).mkdir(exist_ok=True)
(TEXT_OUTPUT := Path("auto_trashcan_label")).mkdir(exist_ok=True)
(EMPTY_OUTPUT := Path("auto_trashcan_empty")).mkdir(exist_ok=True)

for img_path in tqdm(img_paths):
    img_pipeline(
        img_path,
        detect_fn=detect_by_label,
        segment_fn=segmenter.segment_bbox,
        det_output_dir=DET_OUTPUT,
        seg_output_dir=SEG_OUTPUT,
        empty_dir=EMPTY_OUTPUT,
        txt_output_dir=TEXT_OUTPUT,
    )

In [ ]:
# TODO: copy the source images to their own folder, then keep only the labels of well segmented images